In [34]:
import nltk
from nltk.corpus import PlaintextCorpusReader
import re
import pandas as pd
from nltk.corpus import PlaintextCorpusReader, stopwords
from nltk.tokenize import sent_tokenize
from gensim.models.phrases import Phrases, ENGLISH_CONNECTOR_WORDS


In [35]:
# --- Corpus config ---
MDA_FOLDER    =  "../MDA"
# FILENAME_PATTERN = r".*\.txt"
FILENAME_PATTERN = "NVIDIA_([A-Za-z0-9]+(-[A-Za-z0-9]+)+)_[0-9]{4}-[0-9]{2}-[0-9]{2}_MDA\.txt" #only for nvidia 
MIN_SENT_TOKENS  = 5          # discard sentence fragments shorter than this
BIGRAM_MIN_COUNT = 20         # gensim: pair must appear in ≥10 sentences
BIGRAM_THRESHOLD = 3          # gensim: NPMI threshold for joining

In [36]:
# --- Stop words ---
FINANCE_STOPWORDS = {
    "herein", "thereof", "thereto", "therein", "hereby", "pursuant",
    "accordance", "aforementioned", "foregoing", "whereas", "whereby",
    "hereafter", "hereinafter",
    "form", "quarterly", "annual", "report", "filing", "fiscal",
    "quarter", "period", "ended", "condensed", "consolidated",
    "statements", "notes", "refer", "also", "following", "described",
    "set", "forth", "part", "item",
    "incorporated", "reincorporated", "headquartered", "mean", "means",
    "including", "included", "related", "thereto",
    "nvidia", "company", "corporation", "subsidiaries",
}
GEO_STOPWORDS = {
    # --- US States ---
    "alabama", "alaska", "arizona", "arkansas", "california", "colorado",
    "connecticut", "delaware", "florida", "georgia", "hawaii", "idaho",
    "illinois", "indiana", "iowa", "kansas", "kentucky", "louisiana",
    "maine", "maryland", "massachusetts", "michigan", "minnesota",
    "mississippi", "missouri", "montana", "nebraska", "nevada",
    "new_hampshire", "new_jersey", "new_mexico", "new_york", "north_carolina",
    "north_dakota", "ohio", "oklahoma", "oregon", "pennsylvania",
    "rhode_island", "south_carolina", "south_dakota", "tennessee", "texas",
    "utah", "vermont", "virginia", "washington", "west_virginia",
    "wisconsin", "wyoming",

    # --- US State Abbreviations (post-normalisation these are unigrams) ---
    "ca", "de", "tx", "ny", "wa", "nv", "fl", "ga", "il", "ma",

    # --- Major Tech Hub Cities ---
    "san_francisco", "san_jose", "santa_clara","santa","clara","palo_alto", "menlo_park",
    "mountain_view", "sunnyvale", "cupertino", "redwood_city", "fremont",
    "san_diego", "los_angeles", "seattle", "portland", "austin", "boston",
    "new_york_city", "new_york", "chicago", "denver", "atlanta", "miami",

    # --- Common Incorporation / Legal Jurisdictions ---
    "delaware", "nevada", "cayman_islands", "bermuda", "ireland",
    "luxembourg", "singapore", "hong_kong", "switzerland",

    # --- Countries frequently cited in filings ---
    "united_states", "united_kingdom", "china", "japan", "germany",
    "france", "india", "taiwan", "south_korea", "israel", "canada",
    "australia", "netherlands", "sweden", "finland",

    # --- Regions ---
    "north_america", "south_america", "latin_america", "europe",
    "asia_pacific", "middle_east", "apac", "emea", "americas",
}


# --- FLS boundary ---
# Extraction starts at the first match; everything before is dropped

FLS_START_ANCHORS = [
     r"Overview\s+Our\s+Company\s+and\s+Our\s+Businesses",
     r"overview\s+our\s+company",
]
STOPWORDS = set(stopwords.words("english")) | FINANCE_STOPWORDS | GEO_STOPWORDS
_FLS_PATTERN = re.compile("|".join(FLS_START_ANCHORS), re.IGNORECASE)
# --- Table removal: atomic building blocks ---

# Numeric value token: $44,062 / 60.5 % / (12.5) pts / 0.76
_VAL = (
    r'(?:\$\s*)?-?[\d,]+(?:\.\d+)?\s*(?:%|pts)?'
    r'|\([0-9,\.]+\)\s*(?:%|pts)?'
)

# Date token: Apr 27, 2025
_DATE = (
    r'(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)'
    r'\s+\d{1,2},?\s*\d{4}'
)

# Period/change column labels
_PERIOD = (
    r'Three\s+Months\s+Ended'
    r'|Six\s+Months\s+Ended'
    r'|Nine\s+Months\s+Ended'
    r'|Twelve\s+Months\s+Ended'
    r'|Quarter-over-Quarter\s+Change'
    r'|Year-over-Year\s+Change'
    r'|\$\s*Change'
    r'|%\s*Change'
)

# Units header: ($ in millions) / ($ in millions, except per share data)
_UNITS = r'\(\$?\s*[Ii]n\s+(?:millions|billions)[^)]*\)'

# --- Table removal: compiled patterns ---
# Order of application matters — see Cell 2

RE_HEADER_ROW = re.compile(
    r'(?:{date}|{period}|{units})'
    r'(?:\s+(?:{date}|{period}|{units}))*'
    .format(date=_DATE, period=_PERIOD, units=_UNITS),
    re.IGNORECASE
)

RE_DATA_ROW = re.compile(
    r'(?<![.!?]\s)'
    r'[A-Za-z][A-Za-z0-9 \&\(\),\-\.]{{1,50}}'
    r'(?:\s+(?:{val})){{2,}}'
    .format(val=_VAL),
    re.IGNORECASE
)
RE_FOLLOWING_TABLE = re.compile(
    r'[Tt]he following table[^.]+\.',
    re.IGNORECASE
)

RE_ORPHAN_NUMS = re.compile(
    r'(?<!\w)'
    r'(?:{val})(?:\s+(?:{val})){{0,3}}'
    r'(?!\s*\w{{4}})'
    .format(val=_VAL),
    re.IGNORECASE
)

RE_PAGE_NUM = re.compile(r'\b\d{1,3}\b(?=\s+[A-Z])')

# --- Structural noise ---
RE_ITEM_TAG   = re.compile(r'\bItem\s+\d+[A-Z]?\.\s*')
RE_PART_TAG   = re.compile(r'\bPart\s+[IVX]+,?\s*')
RE_COPYRIGHT  = re.compile(r'©?\s*\d{4}\s+NVIDIA[^.]*\.')
RE_WHITESPACE = re.compile(r'\s{2,}')

NGRAM_BLACKLIST_PATTERNS = [

    # --- Punctuation / symbol artifacts ---
    r'^\s*[,;:\-\(\)]+',
    r'[,;:\-\(\)]+\s*$',
    r'^-\w+',
    # --- Repeated tokens ---
    r'\b(\w{4,})_\1\b',              # immediate: california_california
    r'\b(\w{4,})_\w+_\1\b',         # skip-1: april_delaware_april
    r'(\b\w+_\w+)_\w*_?\1',         # bigram chunk repeat: santa_clara_california_santa_clara

    # --- Geographic boilerplate (extend this) ---
    r'\b(santa_clara|san_jose|palo_alto|menlo_park|santa_monica)\b',
    r'\b(california|delaware|nevada|washington|new_york|texas|georgia)\b',
    r'\b(san|santa|los|las|new|mount|fort|north|south|east|west)_\b',  # dangling prefix: san_, santa_
    r'\bsan_(francisco|jose|diego|antonio|mateo|bernardino)\b',
    r'\bsanta_(clara|monica|barbara|ana|cruz|rosa)\b',
    r'\blos_(angeles|altos|gatos)\b',
    r'\bnew_(york|jersey|mexico|hampshire|orleans|haven)\b',
    r'\bfort_(worth|lauderdale|wayne|collins)\b',
    r'\bmount_(view|pleasant)\b',  # mountain_view fragment

    # --- Catch any n-gram that contains 2+ state/city names ---
    r'\b(california|delaware|nevada|washington|texas).*\b(california|delaware|nevada|washington|texas)\b',

    # --- SEC filing structural terms ---
    r'\bnote_financial\b',
    r'\b(form_10|10_k|10_q|8_k)\b',
    r'\b(exhibit|item_\d+|part_[ivx]+)\b',

    # --- Segment boilerplate ---
    r'\b(one|two|three|four|five|reportable)_segment',
    r'\breporting_segment',

    # --- Modal verb fragments ---
    r'\b(entered|continue|remained|could|would|should)_may\b',
    r'\bmay_(continue|remain|result|impact|affect|cause)\b',

    # --- Ordinal / numeric artifacts ---
    r'\b(first|second|third|fourth)_quarter_\w+_quarter\b',
    r'^\d+_',
]


In [37]:
## HELPER FUNCTIONS 

def is_blacklisted(ngram: str) -> bool:
    return any(re.search(pat, ngram, re.IGNORECASE) for pat in NGRAM_BLACKLIST_PATTERNS)

def strip_fls(raw: str) -> str:
    match = _FLS_PATTERN.search(raw)
    if match:
        return raw[match.start():]
    print("  WARNING: FLS anchor not found — using full text.")
    return raw

# --- Table removal ---
def remove_tables(text: str) -> str:
    text = RE_HEADER_ROW.sub(' ', text)
    text = RE_DATA_ROW.sub(' ', text)
    text = RE_FOLLOWING_TABLE.sub(' ', text)
    text = RE_ORPHAN_NUMS.sub(' ', text)
    text = RE_PAGE_NUM.sub(' ', text)
    return text

# --- Full clean ---
def clean(text: str) -> str:
    text = remove_tables(text)
    text = RE_ITEM_TAG.sub(' ', text)
    text = RE_PART_TAG.sub(' ', text)
    text = RE_COPYRIGHT.sub(' ', text)
    text = RE_WHITESPACE.sub(' ', text)
    return text.strip()

# --- Sentence tokenisation ---
def get_sentences(text: str) -> list[str]:
    return [s.strip() for s in sent_tokenize(text)
            if len(s.split()) >= MIN_SENT_TOKENS]

# --- Token normalisation ---
def normalise(sentence: str, remove_stops: bool = False) -> list[str]:
    sentence = sentence.lower()
    sentence = re.sub(r"[^\w\s\-]", "", sentence)
    tokens   = sentence.split()
    tokens   = [t for t in tokens if not re.fullmatch(r'[\d,\.\-\%\$]+', t)]
    if remove_stops:
        tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens

# --- Gensim phrase models ---
def build_phrase_models(all_token_lists):
    bigram  = Phrases(all_token_lists,
                      min_count=BIGRAM_MIN_COUNT,
                      threshold=BIGRAM_THRESHOLD,
                      connector_words=ENGLISH_CONNECTOR_WORDS)
    trigram = Phrases(bigram[all_token_lists],
                      min_count=BIGRAM_MIN_COUNT,
                      threshold=BIGRAM_THRESHOLD,
                      connector_words=ENGLISH_CONNECTOR_WORDS)
    return bigram, trigram

def apply_phrases(tokens, bigram, trigram) -> list[str]:
    phrased = list(trigram[bigram[tokens]])
    # Hard cap: split any n-gram longer than 3 tokens back to unigrams
    result = []
    for tok in phrased:
        parts = tok.split('_')
        if len(parts) <= 3:
            result.append(tok)
        else:
            result.extend(parts)  # bust it back to individual tokens
    return result

# --- Format: words comma-separated within each cell ---
def fmt(tokens: list[str]) -> str:
    return ", ".join(tokens)


In [38]:
# MAIN PIPELINE

def run_pipeline(corpus: PlaintextCorpusReader):
    fileids = corpus.fileids()

    # Pass 1: tokenise all docs, build token store
    print(f"Pass 1: processing {len(fileids)} documents...")
    token_store = {}
    for fid in fileids:
        text = clean(strip_fls(corpus.raw(fid)))
        sents = get_sentences(text)
        token_store[fid] = {
            "raw":   [normalise(s, remove_stops=False) for s in sents],
            "clean": [normalise(s, remove_stops=True)  for s in sents],
        }

    # Train phrase models on clean tokens across full corpus
    all_clean = [tl for d in token_store.values() for tl in d["clean"]]
    print("Training bigram/trigram models...")
    bigram, trigram = build_phrase_models(all_clean)

    # Pass 2: apply phrases + blacklist filter, build dataframe rows
    print("Pass 2: building output...")
    rows_raw, rows_clean = [], []

    for fid in fileids:
        raw_tokens    = token_store[fid]["raw"]
        clean_phrased = [
            [tok for tok in apply_phrases(tl, bigram, trigram)
             if not is_blacklisted(tok)]
            for tl in token_store[fid]["clean"]
        ]

        row_raw   = {"doc": fid}
        row_clean = {"doc": fid}
        for i, (rt, ct) in enumerate(zip(raw_tokens, clean_phrased), start=1):
            row_raw[f"sent_{i}"]   = fmt(rt)
            row_clean[f"sent_{i}"] = fmt(ct)

        rows_raw.append(row_raw)
        rows_clean.append(row_clean)

    def to_df(rows):
        df = pd.DataFrame(rows)
        sent_cols = sorted([c for c in df.columns if c.startswith("sent_")],
                           key=lambda x: int(x.split("_")[1]))
        return df[["doc"] + sent_cols]

    return to_df(rows_raw), to_df(rows_clean)

In [39]:
# --- Run and export ---
nvidia_corpus = PlaintextCorpusReader(MDA_FOLDER, FILENAME_PATTERN)

df_raw, df_clean = run_pipeline(nvidia_corpus)
print(f"Done. Shape: {df_raw.shape}")

with pd.ExcelWriter("mda_processed_sample.xlsx", engine="openpyxl") as writer:
    df_raw.to_excel(writer,   sheet_name="before_stopword_removal", index=False)
    df_clean.to_excel(writer, sheet_name="after_stopword_removal",  index=False)

Pass 1: processing 64 documents...
Training bigram/trigram models...
Pass 2: building output...
Done. Shape: (64, 452)
